# 10 · Pose-based SLAM with scan matching

**Recap** ([theory](https://github.com/Rudip1/learn-state-estimation/blob/main/1_theory/10_pose_graph_slam.md)).
The state is the trajectory only: one node per keyframe, each carrying its lidar scan. **Scan matching** (ICP) turns two
scans into a relative-pose measurement: point-to-point solves each step in closed form, $\theta = \operatorname{atan2}(S_{xy}-S_{yx},
S_{xx}+S_{yy})$; point-to-line minimises distances across the walls and converges much faster. A **pose graph** collects
these measurements as edges with error $e_{ij} = \ominus z_{ij}\oplus(\ominus x_i\oplus x_j)$ and information
$\Omega_{ij} = (J_z\Sigma_zJ_z^T)^{-1}$; Gauss–Newton minimises $\sum e^T\Omega e$ with a sparse $H$ and node 0 fixed. The
front end decides which edges go in; the total $\chi^2$ against its degrees of freedom tells whether one of them is wrong.

In [ ]:
# Installs the module when it is not available yet (e.g. on Colab). Building the C++ takes a few minutes.
try:
    import state_estimation
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-state-estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import state_estimation as se
from state_estimation import plotting

plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6, 4.5)})

## Two scans of the office

The office of this chapter: 16 m × 12 m, split in three by two walls with doorways, plus some clutter. A 360-beam lidar
(range noise 1 cm, 8 m range) takes a scan from two poses 0.6 m and 15° apart. Each scan is a set of points in its own
robot frame; drawn in the world frame with the true poses they overlap.

In [ ]:
walls = se.office_world()
rng = se.Rng(1)
xa, xb = np.array([-5.5, -4.0, 0.1]), np.array([-4.95, -3.8, 0.36])
ref = se.simulate_scan(xa, walls, 360, 2 * np.pi - 1e-3, 8.0, 0.01, rng)
cur = se.simulate_scan(xb, walls, 360, 2 * np.pi - 1e-3, 8.0, 0.01, rng)
truth_ab = se.between(xa, xb)

def draw_walls(ax):
    for w in walls:
        ax.plot([w.a[0], w.b[0]], [w.a[1], w.b[1]], color="0.75", lw=3, zorder=0)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].plot(ref[:, 0], ref[:, 1], ".", ms=2, label="reference (own frame)")
axs[0].plot(cur[:, 0], cur[:, 1], ".", ms=2, label="current (own frame)")
axs[0].set_title("as measured")
draw_walls(axs[1])
for scan, x in ((ref, xa), (cur, xb)):
    w = se.transform_points(x, scan)
    axs[1].plot(w[:, 0], w[:, 1], ".", ms=2)
axs[1].set_title("in the world frame, at the true poses")
for a in axs:
    a.set_aspect("equal"); a.grid(alpha=0.3)
axs[0].legend(fontsize=8); plt.show()
print("true relative pose (x, y, θ):", truth_ab.round(4))

### ✏️ Exercise 1 — the closed-form alignment

Implement `align(P, Q)`, eq. (10.3)–(10.5): the pose $x$ minimising $\sum_i\lVert x\boxplus p_i - q_i\rVert^2$ for paired rows of
`P` and `Q`. Check it against `se.align_point_sets` on noisy pairs.

In [ ]:
def align(P, Q):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
g = np.random.default_rng(0)
P = g.uniform(-3, 3, (40, 2))
Q = se.transform_points(np.array([0.4, -0.2, 0.7]), P) + 0.02 * g.standard_normal((40, 2))
assert np.allclose(align(P, Q), se.align_point_sets(P, Q))
print("✔ closed form matches:", align(P, Q).round(4), "(true pose before noise: [0.4, -0.2, 0.7])")

## ICP: point-to-point against point-to-line

Both start from the same guess, 20 cm and 0.1 rad off. Running them with an increasing iteration limit shows how the
error falls iteration by iteration.

In [ ]:
guess = truth_ab + np.array([0.2, -0.15, 0.1])
def pose_error(x):
    return np.hypot(*(x[:2] - truth_ab[:2])), abs(se.wrap_angle(x[2] - truth_ab[2]))

curves = {}
for name, fn in (("point-to-point", se.icp_point_to_point), ("point-to-line", se.icp_point_to_line)):
    errs = []
    for it in range(1, 31):
        o = se.IcpOptions(); o.max_iterations = it
        errs.append(pose_error(fn(ref, cur, guess, o).pose)[0])
    r = fn(ref, cur, guess)
    curves[name] = errs
    print(f"{name:15s} converged={r.converged} after {r.iterations:2d} iterations; "
          f"error {1000 * pose_error(r.pose)[0]:.1f} mm, {np.rad2deg(pose_error(r.pose)[1]):.3f}°; rmse {1000 * r.rmse:.1f} mm")
for name, e in curves.items():
    plt.semilogy(range(1, 31), e, "o-", ms=3, label=name)
plt.xlabel("iterations"); plt.ylabel("translation error [m]"); plt.legend(); plt.grid(alpha=0.3); plt.show()

Point-to-point keeps a large rmse even at the right pose: the two scans hit the walls at different points, so
every pair is up to half a beam spacing apart. Point-to-line only measures across the walls.

## 🔨 Break it — a bad initial guess

ICP converges to the *nearest* minimum. Start 0.8 rad off: the match lands somewhere wrong, and the covariance it
reports is still small — a wrong match does not announce itself.

In [ ]:
for dth in (0.1, 0.4, 0.8):
    r = se.icp_point_to_line(ref, cur, truth_ab + np.array([0.0, 0.0, dth]))
    et, eth = pose_error(r.pose)
    print(f"guess {dth:.1f} rad off: error {et:.3f} m, {np.rad2deg(eth):6.2f}°; rmse {100 * r.rmse:.1f} cm; "
          f"reported σ_θ {np.rad2deg(np.sqrt(r.covariance[2, 2])):.3f}°")

The rmse of a wrong match is larger, which is why the SLAM front end only accepts matches with a small rmse — and
additionally checks them against odometry (§10.8).

## 🔨 Break it — a corridor

Two long parallel walls: every normal points across the corridor, and $J^TJ$ loses rank. Its smallest eigenvector
is the direction the scans cannot see.

In [ ]:
corridor = [se.Segment(np.array([-50.0, -1.0]), np.array([50.0, -1.0])),
            se.Segment(np.array([-50.0, 1.0]), np.array([50.0, 1.0]))]
for sigma in (0.0, 0.01):
    r_ = se.Rng(6)
    a = se.simulate_scan(np.zeros(3), corridor, 360, 2 * np.pi - 1e-3, 6.0, sigma, r_)
    b = se.simulate_scan(np.array([0.3, 0.1, 0.05]), corridor, 360, 2 * np.pi - 1e-3, 6.0, sigma, r_)
    r = se.icp_point_to_line(a, b, np.array([0.3, 0.1, 0.05]))
    lam, V = np.linalg.eigh(r.hessian)
    print(f"range noise {sigma}: eigenvalues of JᵀJ {lam.round(4)}; weakest direction {V[:, 0].round(3)}; "
          f"estimated x {r.pose[0]:.3f} (true 0.3)")

Noise-free, the smallest eigenvalue is zero and its eigenvector is the corridor axis $x$. With noise the normals tilt
at random and invent a little information along the corridor — the estimate of $x$ there is noise, whatever the
covariance says.

## The pose graph

### ✏️ Exercise 2 — edge error and Jacobians

Implement $e_{ij}$ (10.11) and its Jacobians $A = \partial e/\partial x_i$, $B = \partial e/\partial x_j$ (10.12).

In [ ]:
def edge_error(xi, xj, z):
    # ✏️ your code here
    raise NotImplementedError("exercise")

def edge_jacobians(xi, xj, z):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
xi, xj, z = np.array([1.0, 2.0, 0.4]), np.array([2.5, 1.0, -0.8]), np.array([1.2, -1.0, -1.1])
assert np.allclose(edge_error(xi, xj, z), se.pose_graph_error(xi, xj, z))
A, B = edge_jacobians(xi, xj, z)
A_ref, B_ref = se.pose_graph_jacobians(xi, xj, z)
assert np.allclose(A, A_ref) and np.allclose(B, B_ref)
print("✔ edge error and Jacobians match the C++")

### ✏️ Exercise 3 — one Gauss–Newton step

For a small graph, assemble $H$ and $b$ (10.15) as dense arrays over the free nodes $1..n-1$, solve $H\delta = -b$ and
return the updated poses. One step must equal `optimize(max_iterations=1)`.

In [ ]:
def gauss_newton_step(nodes, edges):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
g = se.PoseGraph()
true_poses = [np.array([0.0, 0, 0]), np.array([1.0, 0, 0.5]), np.array([1.5, 1, 1.6]), np.array([0.5, 1.4, 3.0])]
noise = np.random.default_rng(1)
for k, x in enumerate(true_poses):
    g.add_node(x if k == 0 else x + noise.normal(0, [0.2, 0.2, 0.1]))
for i, j in ((0, 1), (1, 2), (2, 3), (3, 0), (0, 2)):
    g.add_edge(i, j, se.between(true_poses[i], true_poses[j]), np.diag([100.0, 100, 400]))
mine = gauss_newton_step(g.nodes, g.edges)
g.optimize(max_iterations=1)
assert np.allclose(mine, g.nodes)
print("✔ one Gauss–Newton step matches; χ² after one step:", round(g.chi2(), 6))

## SLAM in the office

Two laps; a keyframe every 10 steps (≈ 0.5 m); wheel encoders with the noise of chapter 9. `PoseGraphSlam` builds the
graph keyframe by keyframe: scan-matched odometry edges (gated against wheel odometry), loop closures searched within
1.5 m + 3σ of the current estimate, and a re-optimisation after every closure.

In [ ]:
dd, dt, k = se.DifferentialDrive(0.1, 0.5), 0.1, 2e-4
x0 = np.array([-5.5, -4.0, 0.0])
run = se.simulate_lidar_run(x0, se.office_lap_controls(2, dt), dt, dd, k, walls, 10, 180, 8.0, 0.01, se.Rng(5))
slam = se.PoseGraphSlam(x0, run.scans[0])
for f in range(1, len(run.scans)):
    slam.add_keyframe(run.odometry[f], run.odometry_cov[f], run.scans[f])
graph = slam.graph
dr = se.run_dead_reckoning(x0, np.zeros((3, 3)), run.wheel_travel, dd, k)
truth_k = run.truth[run.keyframes]

def ate(poses):
    return np.sqrt(np.mean(np.sum((poses[:, :2] - truth_k[:, :2]) ** 2, axis=1)))

def chi2_test(g):
    dof = 3 * g.n_edges - 3 * (g.n_nodes - 1)
    return g.chi2(), dof, se.chi2_quantile(0.999, dof)

c2, dof, gate = chi2_test(graph)
print(f"{graph.n_nodes} keyframes, {slam.n_scan_matched} scan-matched odometry edges, {len(slam.loop_closures)} loop closures")
print(f"absolute trajectory error: dead reckoning {ate(dr.poses[run.keyframes]):.3f} m, pose graph {ate(graph.nodes):.3f} m")
print(f"χ² = {c2:.1f} for {dof} degrees of freedom (99.9 % gate {gate:.1f})")

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, poses, title in ((axs[0], dr.poses[run.keyframes], "dead reckoning"), (axs[1], graph.nodes, "pose graph")):
    draw_walls(ax)
    pts = np.vstack([se.transform_points(poses[f], run.scans[f]) for f in range(len(poses))])
    ax.plot(pts[:, 0], pts[:, 1], ".", ms=0.5, alpha=0.5)
    ax.plot(poses[:, 0], poses[:, 1], "-", lw=1)
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title(f"map from {title}")
for a, b in slam.loop_closures:
    axs[1].plot(graph.nodes[[a, b], 0], graph.nodes[[a, b], 1], "g-", lw=0.5)
plt.show()

The information matrix: a band from the odometry chain and two off-diagonal bands from the loop closures of the
second lap.

In [ ]:
plt.spy(np.abs(graph.information_matrix()) > 0, markersize=0.3)
plt.title(f"H: {graph.n_nodes} nodes, {graph.n_edges} edges"); plt.show()
P_last = graph.marginal_covariance(graph.n_nodes - 1)
print("marginal σ of the last keyframe: x %.1f mm, y %.1f mm, θ %.3f°" % (
    1000 * np.sqrt(P_last[0, 0]), 1000 * np.sqrt(P_last[1, 1]), np.rad2deg(np.sqrt(P_last[2, 2]))))

## 🔨 Break it — the edge information shortcut

Passing $\Sigma_z^{-1}$ as the information instead of $(J_z\Sigma_zJ_z^T)^{-1}$ (10.17). On an odometry chain the correct
version reproduces the dead-reckoning covariance exactly; the shortcut does not.

In [ ]:
Q = np.diag([0.01, 0.002, 0.003])
u = np.array([0.5, 0.05, 0.3])
x, P = np.zeros(3), np.zeros((3, 3))
chains = {"(J Σ Jᵀ)⁻¹": se.PoseGraph(), "Σ⁻¹": se.PoseGraph()}
for g_ in chains.values():
    g_.add_node(np.zeros(3))
for step in range(12):
    F, W = se.compose_jacobian_1(x, u), se.compose_jacobian_2(x, u)
    P = F @ P @ F.T + W @ Q @ W.T
    x = se.compose(x, u)
    for name, g_ in chains.items():
        g_.add_node(x)
        g_.add_edge(step, step + 1, u, se.pose_graph_information(u, Q) if name.startswith("(") else np.linalg.inv(Q))
for name, g_ in chains.items():
    g_.optimize()
    err = np.linalg.norm(g_.marginal_covariance(g_.n_nodes - 1) - P) / np.linalg.norm(P)
    print(f"Ω = {name:10s}: relative difference to the dead-reckoning covariance {err:.2e}")

## 🔨 Break it — a false loop closure

A wrong closure with the confidence of a good scan match: keyframe 60 is said to be 0.5 m from where keyframe 20 saw it.

**First in a graph with a single loop** — the first lap's odometry chain closed by one correct loop closure. The false
edge bends the map, and its error is shared with every edge of the loop it creates: the total $\chi^2$ fails its test
clearly, while the false edge's own term carries only part of it. The largest term is still the first suspect;
removing it restores the map.

In [ ]:
first_lap = sorted(c for c in slam.loop_closures if c[1] <= 95)
n1 = max(j for _, j in first_lap)

def one_loop_graph(extra=None):
    g_ = se.PoseGraph()
    for x in graph.nodes[:n1 + 1]:
        g_.add_node(x)
    for e in graph.edges:
        if e.j <= n1 and (e.j - e.i == 1 or (e.i, e.j) == first_lap[0]):
            g_.add_edge(e.i, e.j, e.z, e.information)
    if extra is not None:
        g_.add_edge(*extra)
    g_.optimize(50)
    return g_

def ate_part(g_):
    return np.sqrt(np.mean(np.sum((g_.nodes[:, :2] - truth_k[:g_.n_nodes, :2]) ** 2, axis=1)))

z_false = se.between(truth_k[20], truth_k[60]) + np.array([0.5, 0.0, 0.0])
false_edge = (20, 60, z_false, se.pose_graph_information(z_false, np.diag([0.02**2, 0.02**2, 0.01**2])))
for label, g_ in (("one loop, correct edges", one_loop_graph()), ("one loop + false closure", one_loop_graph(false_edge))):
    c2, dof, gate = chi2_test(g_)
    terms = np.array(g_.edge_chi2()); w = int(np.argmax(terms))
    print(f"{label:25s}: χ² {c2:6.1f} for {dof:2d} dof (gate {gate:5.1f}) -> {'FAILS ' if c2 > gate else 'passes'}; "
          f"ATE {ate_part(g_):.3f} m; largest term {terms[w]:5.1f} on edge {g_.edges[w].i}–{g_.edges[w].j}")
g_ = one_loop_graph(false_edge); g_.remove_edge(int(np.argmax(g_.edge_chi2()))); g_.optimize()
print(f"{'after removing it':25s}: χ² {g_.chi2():6.1f}; ATE {ate_part(g_):.3f} m")

**Then in the full two-lap graph**, with about 80 correct closures: the graph is too stiff to bend, so the map barely
moves, and the false edge has to carry its whole error itself — it stands out on its own.

In [ ]:
full = se.PoseGraph()
for x in graph.nodes:
    full.add_node(x)
for e in graph.edges:
    full.add_edge(e.i, e.j, e.z, e.information)
full.add_edge(*false_edge)
full.optimize()
c2, dof, gate = chi2_test(full)
terms = np.array(full.edge_chi2()); w = int(np.argmax(terms))
print(f"full graph + false closure: χ² {c2:.0f} for {dof} dof (gate {gate:.0f}); ATE {ate(full.nodes):.3f} m; "
      f"largest term {terms[w]:.0f} on edge {full.edges[w].i}–{full.edges[w].j} (per-edge gate {se.chi2_quantile(0.999, 3):.1f})")

## The same graph in GTSAM (optional)

GTSAM's `BetweenFactorPose2` measures the error in the tangent space of $SE(2)$ instead of (10.11); to first order the
two agree, so both optimisers should find practically the same trajectory. The cell is skipped if GTSAM is not
installed (`pip install gtsam`).

In [ ]:
try:
    import gtsam
except ImportError:
    gtsam = None
    print("GTSAM is not installed: skipped")
if gtsam is not None:
    fg, init = gtsam.NonlinearFactorGraph(), gtsam.Values()
    fg.add(gtsam.PriorFactorPose2(0, gtsam.Pose2(*x0), gtsam.noiseModel.Constrained.All(3)))
    start = se.PoseGraph()
    for i in range(graph.n_nodes):
        init.insert(i, gtsam.Pose2(*dr.poses[run.keyframes[i]]))  # both start from dead reckoning
        start.add_node(dr.poses[run.keyframes[i]])
    for e in graph.edges:
        fg.add(gtsam.BetweenFactorPose2(e.i, e.j, gtsam.Pose2(*e.z), gtsam.noiseModel.Gaussian.Information(e.information)))
        start.add_edge(e.i, e.j, e.z, e.information)
    result = gtsam.GaussNewtonOptimizer(fg, init).optimize()
    gt = np.array([[result.atPose2(i).x(), result.atPose2(i).y(), result.atPose2(i).theta()] for i in range(graph.n_nodes)])
    start.optimize(50)
    print(f"largest position difference GTSAM vs this library: {1000 * np.max(np.linalg.norm(gt[:, :2] - start.nodes[:, :2], axis=1)):.3f} mm")
    print(f"ATE: GTSAM {ate(gt):.4f} m, this library {ate(start.nodes):.4f} m")

## What to remember

- Pose-based SLAM estimates the trajectory; the map is the scans drawn at the estimated poses.
- Point-to-line ICP converges in a few iterations; it still needs a good initial guess, and a wrong match reports a
  small covariance.
- A scan sees nothing along a corridor: $J^TJ$ loses rank in that direction.
- The edge information is the inverse covariance of the *error*, $(J_z\Sigma_zJ_z^T)^{-1}$; node 0 must be fixed.
- The front end decides the quality: gate scan matches against odometry, search loops within $r + 3\sigma$, and check
  the total $\chi^2$ against its degrees of freedom.